In [1]:
import pandas as pd

movies = pd.read_csv("data/ml-latest-small/movies.csv")
ratings = pd.read_csv("data/ml-latest-small/ratings.csv")

print("Movies:", movies.shape)
print("Ratings:", ratings.shape)

Movies: (9742, 3)
Ratings: (100836, 4)


In [2]:
print("Movies:")
display(movies.head())

print("\nRatings:")
display(ratings.head())

Movies:


,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy



Ratings:


,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931


In [3]:
print("Missing values in Movies:")
print(movies.isnull().sum())

print("\nMissing values in Ratings:")
print(ratings.isnull().sum())

Missing values in Movies:
movieId    0
title      0
genres     0
dtype: int64

Missing values in Ratings:
userId       0
movieId      0
rating       0
timestamp    0
dtype: int64


In [4]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Replace "|" with spaces
movies["genres_clean"] = movies["genres"].str.replace("|", " ", regex=False)

# Convert genres into numerical features
tfidf = TfidfVectorizer()
genre_matrix = tfidf.fit_transform(movies["genres_clean"])

print("Genre matrix shape:", genre_matrix.shape)

Genre matrix shape: (9742, 24)


In [5]:
from sklearn.metrics.pairwise import cosine_similarity

# Calculate similarity between all movies
similarity_matrix = cosine_similarity(genre_matrix)

print("Similarity matrix shape:", similarity_matrix.shape)

Similarity matrix shape: (9742, 9742)


In [6]:
def recommend_movies(movie_title, n=5):
    movie_index = movies.index[movies["title"] == movie_title][0]

    similarity_scores = list(enumerate(similarity_matrix[movie_index]))
    similarity_scores = sorted(
        similarity_scores,
        key=lambda x: x[1],
        reverse=True
    )

    top_movies = similarity_scores[1:n+1]

    recommendations = []

    for index, score in top_movies:
        recommendations.append({
            "Movie": movies.iloc[index]["title"],
            "Genres": movies.iloc[index]["genres"],
            "Similarity": round(score, 3)
        })

    return pd.DataFrame(recommendations)

In [7]:
recommend_movies("Toy Story (1995)")

,Movie,Genres,Similarity
0,Antz (1998),Adventure|Animation|Children|Comedy|Fantasy,1.0
1,Toy Story 2 (1999),Adventure|Animation|Children|Comedy|Fantasy,1.0
2,"Adventures of Rocky and Bullwinkle, The (2000)",Adventure|Animation|Children|Comedy|Fantasy,1.0
3,"Emperor's New Groove, The (2000)",Adventure|Animation|Children|Comedy|Fantasy,1.0
4,"Monsters, Inc. (2001)",Adventure|Animation|Children|Comedy|Fantasy,1.0


In [8]:
# Create user-movie rating matrix
user_movie_matrix = ratings.pivot_table(
    index="userId",
    columns="movieId",
    values="rating"
).fillna(0)

print("User-Movie Matrix:", user_movie_matrix.shape)

User-Movie Matrix: (610, 9724)


In [9]:
from sklearn.metrics.pairwise import cosine_similarity

user_similarity = cosine_similarity(user_movie_matrix)

print("User similarity matrix:", user_similarity.shape)

User similarity matrix: (610, 610)


In [10]:
def collaborative_recommendations(user_id, n=5):
    user_index = user_movie_matrix.index.get_loc(user_id)

    similar_users = list(enumerate(user_similarity[user_index]))
    similar_users = sorted(
        similar_users,
        key=lambda x: x[1],
        reverse=True
    )[1:6]

    scores = {}

    for index, similarity in similar_users:
        ratings_by_user = user_movie_matrix.iloc[index]

        for movie_id, rating in ratings_by_user.items():
            if rating > 0:
                scores[movie_id] = scores.get(movie_id, 0) + similarity * rating

    watched = user_movie_matrix.loc[user_id]
    scores = {
        movie_id: score
        for movie_id, score in scores.items()
        if watched.get(movie_id, 0) == 0
    }

    top_movies = sorted(scores.items(), key=lambda x: x[1], reverse=True)[:n]

    recommendations = []

    for movie_id, score in top_movies:
        title = movies.loc[movies["movieId"] == movie_id, "title"].values[0]
        recommendations.append(title)

    return recommendations

In [11]:
collaborative_recommendations(1)

['Aliens (1986)',
 'Hunt for Red October, The (1990)',
 'Blade Runner (1982)',
 'Terminator 2: Judgment Day (1991)',
 'Die Hard (1988)']